# Sampling SFT — source preparation

Ticket 01 · **CPU only** · no generation or training.

Verify saved gold-SFT and GRPO identities, then save a restorable experiment plan.
A blocked audit lists missing or conflicting evidence; it does not select fallback settings.
Run this notebook from a checkout containing the Sampling SFT commands. Cloud setup and paid stages are added in later tickets.


## 1. Select the checkout and artifact root

Use the existing artifact tree. For Colab CPU, mount Drive first; set `REPO_DIR` to your checked-out release.


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path.cwd().resolve()
if not (REPO_DIR / "pilot_eval").is_dir():
    REPO_DIR = REPO_DIR.parent
if not (REPO_DIR / "pilot_eval").is_dir():
    raise RuntimeError("Set REPO_DIR to the checked-out spar-pilot repository.")

ARTIFACT_ROOT = REPO_DIR / "artifacts"
PLAN_NAME = "sampling-sft-v1"
# In Colab CPU, uncomment before setting ARTIFACT_ROOT:
# from google.colab import drive
# drive.mount("/content/drive")
# ARTIFACT_ROOT = Path("/content/drive/MyDrive/SPAR/spar-pilot/artifacts")


## 2. Select the existing source configurations

Gold may be its original `sft.config.json` or the prepared SFT activation configuration that explicitly points to it.
GRPO uses its existing `activation.prepared.json`. Paths are relative to the artifact root; no directories are guessed from evaluation reports.


In [ ]:
SFT_SOURCE = "plans/pilot2-sft-fp32-l4-batch2-v1/sft.config.json"
GRPO_SOURCE = "plans/pilot4-grpo-fp32-v1-writes-v1/activation.prepared.json"

selection_path = ARTIFACT_ROOT / "plans" / PLAN_NAME / "source-selection.json"
selection = {"sft": SFT_SOURCE, "grpo": GRPO_SOURCE}
selection_path.parent.mkdir(parents=True, exist_ok=True)
if selection_path.exists() and json.loads(selection_path.read_text()) != selection:
    raise RuntimeError("Saved selection differs. Use a new PLAN_NAME to preserve the old plan.")
if not selection_path.exists():
    selection_path.write_text(json.dumps(selection, indent=2) + "\n")

def run_command(operation, *arguments):
    return subprocess.run(
        [sys.executable, "-u", "-m", "pilot_eval", operation,
         "--output-root", str(ARTIFACT_ROOT), *map(str, arguments)],
        cwd=REPO_DIR, check=True,
    )


## 3. Prepare the immutable plan

Saves the selected paths and approved recipe. Existing compatible plans are reused.


In [ ]:
run_command("sampling-sft-prepare", "--selection", selection_path, "--name", PLAN_NAME)
PLAN_PATH = ARTIFACT_ROOT / "plans" / PLAN_NAME / "sampling-sft.plan.json"


## 4. Audit the original sources

Checks the 512 training examples, 150 held-out identities, saved masks/order, original L4 runtime, training completion and all five adapter checkpoints.
Raw historical files are read only. Output is `source-audited` or `blocked`; neither means engine qualification or experiment execution.


In [ ]:
run_command("sampling-sft-audit", "--config", PLAN_PATH)


## 5. Verify and restore

After a reconnect, run cells 1–2, then this cell. The saved plan determines the paths; no generation is restarted.
If source contents or availability changed, preserve this plan and prepare a new named one.


In [ ]:
PLAN_PATH = ARTIFACT_ROOT / "plans" / PLAN_NAME / "sampling-sft.plan.json"
run_command("sampling-sft-status", "--config", PLAN_PATH)
run_command("sampling-sft-verify", "--config", PLAN_PATH)

config = json.loads(PLAN_PATH.read_text())
audit_path = ARTIFACT_ROOT / config["run_path"] / "results" / "source-audit.json"
audit = json.loads(audit_path.read_text())
print("Audit:", audit_path)
print("Sources ready:", audit["source_ready"])
print("Next:", "budget/lifecycle implementation" if audit["source_ready"] else "resolve the listed source blockers in a new plan")


## Current boundary

Implemented here: source preparation, audit and CPU restoration.

Pending: budget/lifecycle controls, engine settings and qualification, pilot replay, full generation, review, data gate and separately approved endpoint training.
The first paid job requires an explicit go-ahead after preflight. No baseline, evaluation or internal measurement runs from this notebook.
